read dde from kn.her.all, first column includes two single mutations aa and site information, second column is dde, third is DE_double, fourth is first de, fifth is second de

# 1. read dde from file


In [77]:
import pandas as pd
# Try reading the file with space as a delimiter
df = pd.read_csv('../data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])
# Split the 'Mutation' column into 'First_mutation' and 'Second_mutation'
df[['First_mutation', 'Second_mutation']] = df['Mutation'].str.split('-', expand=True)
# Display the DataFrame
print(df)


        Mutation       DDE  DE_double  First_DE  Second_DE First_mutation  \
0        B1A-C2A  7.232628 -16.029196 -7.656878  -8.055814            B1A   
1        B1A-C2B  2.924126 -17.516587 -7.656878 -11.322448            B1A   
2        B1A-C2D  3.931540 -15.343873 -7.656878  -8.569091            B1A   
3        B1A-C3A  6.683204 -16.250602 -7.656878  -7.958827            B1A   
4        B1A-C3B  2.468630 -15.079575 -7.656878  -8.571916            B1A   
...          ...       ...        ...       ...        ...            ...   
43654  B98C-B99C -0.016739 -18.090722 -6.890913 -11.621474           B98C   
43655  B98C-B99D -0.002944 -21.403432 -6.890913 -15.011783           B98C   
43656  B98D-B99A  0.241060 -12.673526 -8.468545  -4.999852           B98D   
43657  B98D-B99C -0.005845 -19.652754 -8.468545 -11.621474           B98D   
43658  B98D-B99D  0.006986 -22.964501 -8.468545 -15.011783           B98D   

      Second_mutation  
0                 C2A  
1                 C2B  
2  

/var/folders/17/rj19bvws2qscyfjmb7m44zmm0000gn/T/ipykernel_53838/1655497556.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv('../data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])


In [78]:
# Filter rows for D148B and C140D
filtered_dde = df[(df['First_mutation'] == 'D148B') & (df['Second_mutation'] == 'C140D') |
                  (df['First_mutation'] == 'C140D') & (df['Second_mutation'] == 'D148B')]

# Print the DDE values
print(filtered_dde)

Empty DataFrame
Columns: [Mutation, DDE, DE_double, First_DE, Second_DE, First_mutation, Second_mutation]
Index: []


# 2. index each sequence by its mutations compared to in.consensus.reduce4.seq

read in from ../data/in.reduce4.seq, there are 1220 sequences, create a df, with last column as mutations: with a list of mutaiton that occored compared to ../data/in.consensus.reduce4.seq the mutations are in format for example: [D148B, C140D, ...] where D and C notes the wildtime from consensus at pos 148 and 140

In [79]:
# Read the consensus sequence
with open('../data/pr.consensus.reduce4.seq', 'r') as f:
    consensus_sequence = f.read().strip()

# Read the 1220 sequences
sequences = []
with open('../data/pr.exper.reduce4.seq', 'r') as f:
    for line in f:
        sequences.append(line.strip())

# Create a DataFrame to store sequences and their mutations
sequence_df = pd.DataFrame({'Sequence': sequences})

# Function to identify mutations compared to the consensus sequence
def find_mutations(sequence, consensus):
    mutations = []
    for i, (seq_residue, cons_residue) in enumerate(zip(sequence, consensus), start=1):
        if seq_residue != cons_residue:
            mutations.append(f"{cons_residue}{i}{seq_residue}")
    return mutations

# Add a column for mutations
sequence_df['Mutations'] = sequence_df['Sequence'].apply(lambda seq: find_mutations(seq, consensus_sequence))
sequence_df['Mutations_count'] = sequence_df['Mutations'].apply(len)
# Display the DataFrame
print(sequence_df)

                                               Sequence  \
0     BCCDDDCDCCDDDDDDABCDABCABDBACBDCAACCBBCABACBCC...   
1     BCCDDDCDCDDDDCDDABCDABCABDBACBDDAACDBBCABACBCD...   
2     BCCDDDCDCDDDDCCDABCDABCABDBACBDDAADDBBCABACBCD...   
3     BCCDDDCDCCDDDCCDABCDABCDBDBACBDCAADDCBCABACBCC...   
4     BCCDDDCDCDDDDCCDABCDABCABDBACBDCAADDCBCABACBCC...   
...                                                 ...   
5705  BCCDDDCDCDDDDCDDABCDABCABDBACBDCAADDBBCABABBCC...   
5706  BCCCDDCDCDDDDCDCABCDABCABDBACBDCAADDCBCABACBCC...   
5707  BCCDDDCDCDDBDDDDABCDABCABDBACBDCAADDABCABACBCC...   
5708  BCCDDDCDCDDDDDDDABCDABCABDBACBDCAADDCBCABACBCC...   
5709  BCCDDDCDCCDDDCDDABCDABCDBDBACBDCAACDBBCABADBCD...   

                                              Mutations  Mutations_count  
0     [D10C, C14D, D35C, D36C, C37B, C54D, A63D, B64...               12  
1     [C32D, D35C, C37B, C46D, A47B, A63D, C73B, D77...               10  
2     [D15C, C32D, C37B, C46D, A47B, A63D, C82B, A92...           

# 3. J matrix and delta e definition

## 3.1 J matrix

In [80]:
import numpy as np

#dictionary of J matrix
J_dict = {}

# Load the J matrix from the downloaded file
J = np.load('../data/J_PR.npy')

row = 0
# Determine the largest position in the 'Mutation' column of the dataframe
max_position = max(
    int(mutation[1:-1]) for mutation in df['First_mutation'].tolist() + df['Second_mutation'].tolist()
)
# print(max_position+1)
# Update the range to use the largest position
for pos1 in range(1, max_position + 1):
    for pos2 in range(pos1 + 1, max_position + 1):
        for i, aa1 in enumerate(['A', 'B', 'C', 'D']):
            for j, aa2 in enumerate(['A', 'B', 'C', 'D']):
                col = i * 4 + j
                J_dict[(pos1, pos2, aa1, aa2)] = J[row, col]
                J_dict[(pos2, pos1, aa2, aa1)] = J[row, col]
        row += 1

print(f"Dictionary created with {len(J_dict)} entries")

Dictionary created with 155232 entries


## 3.2 define delta e

In [81]:
# Define delta E calculation
def calculate_delta_e(position, old_amino_acid, new_amino_acid, seq, J_dict):
    # E(old_amino_acid)
    energy_old = 0
    for other_pos in range(1, max_position+1):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - 1]  # Access the first sequence in sequence_list
        energy_old += J_dict.get((position, other_pos, old_amino_acid, other_aa), 0)

    # E(new_amino_acid)
    energy_new = 0
    for other_pos in range(1, max_position+1):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - 1]  # Access the first sequence in sequence_list
        energy_new += J_dict.get((position, other_pos, new_amino_acid, other_aa), 0)

    delta_e = energy_old - energy_new
    # print(f"E({old_amino_acid}) at {position}: {energy_old}")
    # print(f"E({new_amino_acid}) at {position}: {energy_new}")
    # print(f"Delta E for {old_amino_acid}{position}{new_amino_acid}: {delta_e}")
    return delta_e

# # Example usage
# position = 140
# old_amino_acid = 'C'
# new_amino_acid = 'D'
# calculate_delta_e(position, old_amino_acid, new_amino_acid, sequence_list, J_dict)

In [82]:
# define dm12
def calculate_dm12 (pos1, old_amino_acid1, new_amino_acid1, pos2, old_amino_acid2, new_amino_acid2, seq, J_dict):
    energy_old = 0
    energy_new = 0

# old energy
    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_old += J_dict.get((pos1, pos2, old_amino_acid1, old_amino_acid2), 0)
        else:
            energy_old += J_dict.get((pos1, other_pos, old_amino_acid1, other_aa), 0)

    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1]
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            # energy_old += J_dict.get((pos2, pos1, old_amino_acid2, old_amino_acid1), 0)
            continue
        else:
            energy_old += J_dict.get((pos2, other_pos, old_amino_acid2, other_aa), 0)

# new energy
    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_new += J_dict.get((pos1, pos2, new_amino_acid1, new_amino_acid2), 0)
        else:
            energy_new += J_dict.get((pos1, other_pos, new_amino_acid1, other_aa), 0)

    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            # energy_new += J_dict.get((pos2, pos1, new_amino_acid2, new_amino_acid1), 0)
            continue
        
        else:
            energy_new += J_dict.get((pos2, other_pos, new_amino_acid2, other_aa), 0)
    
    return energy_old - energy_new




## 3.3 flip on 1220

run de on all 1220 sequences with D148B, C140D (C140D, D148B), if the consensus vs one of 1220 sequences where the DE of each mutation changes its size compare to others  ( DE -DE sign change from consensus where the sign change means the sign of de D148B- deC140D sign change), record that sequence with its mutaitons and mutation len

In [83]:
# Define the mutations as input parameters
mutation1 = 'A1B'
mutation2 = 'B2C'

# Extract position and amino acid information from the mutations
pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

# List to store sequences with sign change
sequences_with_sign_change = []

# Calculate delta E for the consensus sequence
de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

print("-" * 50)
print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
print("-" * 50)

# Iterate through all sequences
for index, row in sequence_df.iterrows():
    sequence = row['Sequence']
    mutations = row['Mutations']
    mutation_count = row['Mutations_count']
    
    # Skip sequences without the specified mutations
    if mutation1 not in mutations or mutation2 not in mutations:
        # continue
        pass
    
    # Calculate delta E for the specified mutations
    de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
    de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
    
    # Check for sign change
    if (de_mutation1 - de_mutation2) * (de_mutation1_consensus - de_mutation2_consensus) < 0:
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)
        min_dm1_dm2 = min(de_mutation1, de_mutation2)
        max_dm1_dm2 = max(de_mutation1, de_mutation2)
        max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
        
        sequences_with_sign_change.append({
            'Sequence': sequence,
            'Mutations': mutations,
            'Mutation_count': mutation_count,
            'dm1': de_mutation1,
            'dm2': de_mutation2,
            'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
            'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
        })

# Create a DataFrame to store the results
sign_change_df = pd.DataFrame(sequences_with_sign_change)

# Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
sign_change_df = sign_change_df.sort_values(by='Dm1m2-min(dm1,dm2)', ascending=False)

# Print the results
for _, row in sign_change_df.iterrows():
    print(f"dm1: {row['dm1']}, dm2: {row['dm2']}")
    print(f"Mutations: {row['Mutations']}")
    print(f"Mutation Count: {row['Mutation_count']}")
    print(f"Dm1m2-min(dm1,dm2): {row['Dm1m2-min(dm1,dm2)']}")
    print("-" * 50)


--------------------------------------------------
Consensus delta E for A1B: 7.6568756103515625, B2C: 11.322447776794434
Consensus dm12: 16.055199
--------------------------------------------------
dm1: 8.512626647949219, dm2: 8.070003509521484
Mutations: ['D10C', 'D15C', 'B18A', 'C19B', 'D20A', 'C46D', 'A63D', 'A71B', 'C72D', 'D77C', 'A84B', 'B85C', 'B88A', 'C90B']
Mutation Count: 14
Dm1m2-min(dm1,dm2): 5.5885009765625
--------------------------------------------------
dm1: 8.398787498474121, dm2: 7.288920879364014
Mutations: ['D10A', 'D13B', 'D15C', 'D20C', 'D35C', 'D36A', 'C45B', 'C46B', 'C62D', 'A63D', 'A71C', 'B88A', 'C90B', 'A93B']
Mutation Count: 14
Dm1m2-min(dm1,dm2): 5.474661350250244
--------------------------------------------------
dm1: 8.398787498474121, dm2: 7.288920879364014
Mutations: ['D10A', 'D13B', 'D15C', 'D20C', 'D35C', 'D36A', 'C45B', 'C46B', 'C62D', 'A63D', 'A71C', 'B88A', 'C90B', 'A93B']
Mutation Count: 14
Dm1m2-min(dm1,dm2): 5.474661350250244
-----------------

## 3.4 compensate on 1220

In [84]:
# Define the mutations as input parameters
mutation1 = 'A1B'
mutation2 = 'B2C'

# Extract position and amino acid information from the mutations
pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

# List to store sequences with sign change
sequences_with_sign_change = []

# Calculate delta E for the consensus sequence
de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

print("-" * 50)
print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
print("-" * 50)

# Iterate through all sequences
for index, row in sequence_df.iterrows():
    sequence = row['Sequence']
    mutations = row['Mutations']
    mutation_count = row['Mutations_count']
    
    # Skip sequences without the specified mutations
    if mutation1 not in mutations or mutation2 not in mutations:
        # continue
        pass
    
    # Calculate delta E for the specified mutations
    de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
    de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
    dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

    # Check for sign change
    if dm12 > de_mutation1 or dm12 > de_mutation2:
        min_dm1_dm2 = min(de_mutation1, de_mutation2)
        max_dm1_dm2 = max(de_mutation1, de_mutation2)
        max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
        
        sequences_with_sign_change.append({
            'Sequence': sequence,
            'Mutations': mutations,
            'Mutation_count': mutation_count,
            'dm1': de_mutation1,
            'dm2': de_mutation2,
            'dm12': dm12,
            'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
            'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
        })

# Create a DataFrame to store the results
sign_change_df = pd.DataFrame(sequences_with_sign_change)

# Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
sign_change_df = sign_change_df.sort_values(by='Dm1m2-min(dm1,dm2)', ascending=False)

# Print the results
for _, row in sign_change_df.iterrows():
    print(f"dm1: {row['dm1']}, dm2: {row['dm2']}")
    print(f"dm12: {row['dm12']}")
    print(f"Mutations: {row['Mutations']}")
    print(f"Mutation Count: {row['Mutation_count']}")
    print(f"Dm1m2-min(dm1,dm2): {row['Dm1m2-min(dm1,dm2)']}")
    print("-" * 50)

--------------------------------------------------
Consensus delta E for A1B: 7.6568756103515625, B2C: 11.322447776794434
Consensus dm12: 16.055199
--------------------------------------------------
dm1: 0.3504047393798828, dm2: 11.455327987670898
dm12: 16.114242553710938
Mutations: ['C2A', 'D10A', 'C14D', 'D35C', 'C37A', 'A63D', 'D77C', 'A93B']
Mutation Count: 8
Dm1m2-min(dm1,dm2): 15.763837814331055
--------------------------------------------------
dm1: 0.4729957580566406, dm2: 7.851139068603516
dm12: 15.556763648986816
Mutations: ['B1A', 'C2A', 'D10C', 'B30D', 'A93B']
Mutation Count: 5
Dm1m2-min(dm1,dm2): 15.083767890930176
--------------------------------------------------
dm1: -6.226642608642578, dm2: 6.924405097961426
dm12: 7.930393218994141
Mutations: ['B1A', 'C2A', 'C3A', 'D35C', 'C37B', 'A63C', 'B64C', 'B65D', 'C82D']
Mutation Count: 9
Dm1m2-min(dm1,dm2): 14.157035827636719
--------------------------------------------------
dm1: -6.2784576416015625, dm2: 6.740981101989746
dm1

## 3.5 Antagonistic interactions

In [85]:
# Define the mutations as input parameters
mutation1 = 'A1B'
mutation2 = 'C90D'

# Extract position and amino acid information from the mutations
pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

# List to store sequences with sign change
sequences_with_sign_change = []

# Calculate delta E for the consensus sequence
de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

print("-" * 50)
print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
print("-" * 50)

# Iterate through all sequences
for index, row in sequence_df.iterrows():
    sequence = row['Sequence']
    mutations = row['Mutations']
    mutation_count = row['Mutations_count']
    
    # Skip sequences without the specified mutations
    if mutation1 not in mutations or mutation2 not in mutations:
        # continue
        pass
    
    # Calculate delta E for the specified mutations
    de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
    de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
    dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

    # Check for sign change
    if dm12 < de_mutation1 and dm12 < de_mutation2:
        min_dm1_dm2 = min(de_mutation1, de_mutation2)
        max_dm1_dm2 = max(de_mutation1, de_mutation2)
        max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
        
        sequences_with_sign_change.append({
            'Sequence': sequence,
            'Mutations': mutations,
            'Mutation_count': mutation_count,
            'dm1': de_mutation1,
            'dm2': de_mutation2,
            'dm12': dm12,
            'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
            'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
        })

# Create a DataFrame to store the results
sign_change_df = pd.DataFrame(sequences_with_sign_change)

# Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
sign_change_df = sign_change_df.sort_values(by='Dm1m2-min(dm1,dm2)', ascending=False)

# Print the results
for _, row in sign_change_df.iterrows():
    print(f"dm1: {row['dm1']}, dm2: {row['dm2']}")
    print(f"dm12: {row['dm12']}")
    print(f"Mutations: {row['Mutations']}")
    print(f"Mutation Count: {row['Mutation_count']}")
    print(f"Dm1m2-min(dm1,dm2): {row['Dm1m2-min(dm1,dm2)']}")
    print("-" * 50)


--------------------------------------------------
Consensus delta E for A1B: 7.6568756103515625, C90D: -15.507952690124512
Consensus dm12: -7.8510733
--------------------------------------------------
dm1: -5.648978233337402, dm2: -15.503101348876953
dm12: -21.133337020874023
Mutations: ['B1A', 'C2A', 'C3A', 'D12B', 'C14D', 'D35C', 'D36C', 'C62D', 'A63D', 'C67A']
Mutation Count: 10
Dm1m2-min(dm1,dm2): -5.63023567199707
--------------------------------------------------
dm1: -5.90135383605957, dm2: -15.028667449951172
dm12: -20.911277770996094
Mutations: ['B1A', 'C2A', 'C3A', 'C62D', 'A63D', 'B64A']
Mutation Count: 6
Dm1m2-min(dm1,dm2): -5.882610321044922
--------------------------------------------------
dm1: -5.920185089111328, dm2: -15.194114685058594
dm12: -21.095558166503906
Mutations: ['B1A', 'C2A', 'C3A', 'C37A', 'A61C', 'A63D', 'B64A', 'D77C', 'C82B', 'A93B']
Mutation Count: 10
Dm1m2-min(dm1,dm2): -5.9014434814453125
--------------------------------------------------
dm1: -5.95